In [1]:
import duckdb

conn = duckdb.connect("main.db")

In [9]:
city_priority_query = """
WITH vehicle_proxy AS (
    SELECT
        state,
        SUM(value) AS total_vehicles
    FROM vehicle_registrations
    WHERE year = (SELECT MAX(year) FROM vehicle_registrations)
    GROUP BY state
),

state_growth AS (
    SELECT
        state,
        AVG(CASE WHEN year = 2011 AND gender = 'Total' THEN value * 1000 END) AS pop_2011,
        AVG(CASE WHEN year = 2025 AND gender = 'Total' THEN value * 1000 END) AS pop_2025,
        (AVG(CASE WHEN year = 2025 AND gender = 'Total' THEN value * 1000 END) / 
         AVG(CASE WHEN year = 2011 AND gender = 'Total' THEN value * 1000 END)) AS growth_multiplier
    FROM population_data
    GROUP BY state
),

census_cities AS (
    SELECT
        CASE WHEN City = 'Gurgaon' THEN 'Gurugram' ELSE City END AS City,
        CASE 
            WHEN City = 'Delhi' THEN 'Delhi'
            WHEN State = 'Uttar Pradesh' THEN 'Uttar Pradesh'
            WHEN State = 'Haryana' THEN 'Haryana'
            WHEN State = 'Bihar' THEN 'Bihar'
            WHEN State = 'Rajasthan' THEN 'Rajasthan'
            WHEN State = 'Madhya Pradesh' THEN 'Madhya Pradesh'
            ELSE State
        END AS state_normalized,
        Metro_Population AS metro_pop_2011
    FROM indian_cities
    WHERE City IN ('Ghaziabad', 'Noida', 'Delhi', 'Faridabad', 
                   'Gurgaon', 'Patna', 'Muzaffarpur')
       OR City LIKE '%Greater Noida%'
       OR City LIKE '%Baghpat%'
       OR City LIKE '%Bhiwadi%'
       OR City LIKE '%Singrauli%'
       OR City LIKE '%Chhapra%'
),

city_population AS (
    SELECT
        City AS area,
        state_normalized AS state,
        metro_pop_2011 * growth_multiplier AS pop_2025
    FROM census_cities
    JOIN state_growth
    ON census_cities.state_normalized = state_growth.state
),

city_metrics AS (
    SELECT
        a.area,
        a.state,
        ROUND(AVG(a.aqi_value), 0) AS avg_aqi,
        COUNT(CASE WHEN a.air_quality_status IN ('Poor','Very Poor','Severe') THEN 1 END) * 100.0 / COUNT(*) AS pct_bad_days,
        COUNT(CASE WHEN a.air_quality_status = 'Severe' THEN 1 END) AS severe_days,
        COUNT(*) AS total_days
    FROM air_quality a
    GROUP BY a.area, a.state
    HAVING pct_bad_days >= 40
)

SELECT
    cm.area,
    cm.state,
    cm.avg_aqi,
    ROUND(cm.pct_bad_days, 1) AS pct_bad_days,
    cm.severe_days,
    ROUND(cp.pop_2025 / 1000000, 2) AS pop_millions,
    ROUND(v.total_vehicles / 1000000, 2) AS vehicles_millions,
    -- Risk score: 40% severity + 30% population + 30% income proxy
    ROUND(cm.pct_bad_days * 0.4 + cm.avg_aqi/5 * 0.3 + (cp.pop_2025/1000000) * 0.15 + (v.total_vehicles/1000000) * 0.15, 1) AS risk_score,
    CASE WHEN cm.pct_bad_days >= 40 THEN 'Launch Priority' 
         WHEN cm.pct_bad_days >= 20 THEN 'Phase 2'
         ELSE 'Monitor' END AS tier
FROM city_metrics cm
LEFT JOIN city_population cp ON cm.area = cp.area
LEFT JOIN vehicle_proxy v ON cm.state = v.state
WHERE cm.total_days >= 1000
ORDER BY risk_score DESC
LIMIT 25 """

conn.execute(city_priority_query).fetch_df().to_csv("/workspaces/Product-Market-Fit-Analysis---Case-Study/Output_Dataset/priority_cities.csv", index=False)

In [10]:
seasonal_aqi_query = """
WITH seasonal_data AS (
    SELECT
        area,
        state,
        date,
        CASE WHEN EXTRACT('month' FROM date) IN (11,12,1,2) THEN 'Winter'
             WHEN EXTRACT('month' FROM date) IN (3,4,5,6) THEN 'Summer'
             WHEN EXTRACT('month' FROM date) IN (7,8,9,10) THEN 'Monsoon'
        END AS season,
        air_quality_status,
        aqi_value
    FROM air_quality
),

city_seasonal_stats AS (
    SELECT
        area,
        season,
        COUNT(*) AS total_days,
        AVG(aqi_value) AS avg_aqi,
        COUNT(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 END) AS bad_days,
        COUNT(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 END) * 100.0 / COUNT(*) AS pct_bad_days
    FROM seasonal_data
    GROUP BY area, season
),

city_overall_risk AS (
    SELECT
        area,
        AVG(CASE 
             WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
             THEN 1 ELSE 0 END) * 100 AS overall_pct_bad
    FROM air_quality
    GROUP BY area
)

SELECT
    'High Risk Cities' AS segment,
    season,
    ROUND(AVG(pct_bad_days), 1) AS avg_pct_bad,
    ROUND(AVG(avg_aqi), 0) AS avg_aqi,
    COUNT(DISTINCT area) AS num_cities
FROM city_seasonal_stats
JOIN city_overall_risk USING (area)
WHERE overall_pct_bad >= 40
GROUP BY season

UNION ALL

SELECT 'All India', season, -- for comparison
    AVG(pct_bad_days), AVG(avg_aqi), COUNT(DISTINCT area)
FROM city_seasonal_stats
GROUP BY season
"""

conn.execute(seasonal_aqi_query).fetch_df().to_csv("/workspaces/Product-Market-Fit-Analysis---Case-Study/Output_Dataset/aqi_seasonality.csv", index=False)

In [8]:
competitor_matrix_query = """ 
-- Data researched using Perplexity
SELECT 
    brand,
    model,
    price_range,
    has_hepa, 
    has_carbon, 
    has_pm25_sensor, 
    has_app, 
    has_auto_mode, 
    has_sleep_mode, 
    cadr_range,
    market_position -- Budget/Mid/Premium
FROM (VALUES
    ('Xiaomi', 'Air Purifier 4 Lite', '11-13k', 1, 0, 1, 1, 1, 1, '300+', 'Budget'),
    ('Honeywell', 'Air Touch V1', '8-10k', 1, 1, 0, 0, 1, 0, '300-350', 'Budget'),
    ('Coway', 'AirMega 150', '16-19k', 1, 1, 0, 0, 1, 0, '250-300', 'Mid'),
    ('Philips', 'AC1711', '16-20k', 1, 0, 1, 0, 1, 1, '350-400', 'Mid'),
    ('Dyson', 'TP10', '38-55k', 1, 0, 1, 1, 1, 1, '500+', 'Premium'),
    ('AirPure', 'Premium (Planned)', '18-22k', 1, 1, 1, 1, 1, 1, '350+', 'Mid-Premium')
) AS t(brand, model, price_range, has_hepa, has_carbon, has_pm25_sensor, 
         has_app, has_auto_mode, has_sleep_mode, cadr_range, market_position)
"""

conn.execute(competitor_matrix_query).fetch_df().to_csv("/workspaces/Product-Market-Fit-Analysis---Case-Study/Output_Dataset/competitor_features_matrix.csv", index=False)